### LangChain chatbot with RAG - experiments

Imports and evironment

In [ ]:
import random

from dotenv import load_dotenv
from langchain.agents import create_agent  # ← NOWE API!
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load environment variables from .env file with OPENAI_API_KEY
load_dotenv()

True

#### 1. Documents preparation

Dictionary of terms

In [2]:
documents = []

# Load dictionary PDF
dict_loader = PyPDFLoader("../documents/slownik_pojec.pdf")
dict_pages = dict_loader.load()
for page in dict_pages:
    documents.append(
        Document(
            page_content=page.page_content,
            metadata={
                "source": "slownik_pojec.pdf",
                "page": page.metadata.get("page", ""),
                "language": "pl",
                "document_type": "dictionary_of_terms",
                "publication_year": "2017",
            },
        )
    )
print(f"Loaded {len(dict_pages)} pages from slownik_pojec.pdf")

Loaded 10 pages from slownik_pojec.pdf


Mortgage Loan Act

In [3]:
# Load law PDF

law_loader = PyPDFLoader("../documents/ustawa.pdf")
law_pages = law_loader.load()
for page in law_pages:
    documents.append(
        Document(
            page_content=page.page_content,
            metadata={
                "source": "ustawa.pdf",
                "page": page.metadata.get("page", ""),
                "language": "pl",
                "document_type": "law_act",
                "publication_year": "2024",
            },
        )
    )
print(f"Loaded {len(law_pages)} pages from ustawa.pdf")

Loaded 47 pages from ustawa.pdf


In [4]:
# Preview of documents

print("=== DOCUMENTS PREVIEW ===")
random_docs = random.sample(documents, min(3, len(documents)))

for i, doc in enumerate(random_docs):
    print(f"\n--- Random example {i+1} ---")
    print(f"Source: {doc.metadata['source']}")
    print(f"Page: {doc.metadata['page']}")
    print(f"Content (first 500 characters):")
    print(doc.page_content[:500] + "..." if len(doc.page_content) > 500 else doc.page_content)
    print(f"{'-'*60} \n")

=== DOCUMENTS PREVIEW ===

--- Random example 1 ---
Source: slownik_pojec.pdf
Page: 6
Content (first 500 characters):
Rata annuitetowa - Rata kredytu o stałej wysokości, w której zmienia się proporcja
między kapitałem a odsetkami.
Rata kapitałowa - Część raty kredytu przeznaczona na spłatę kapitału.
Rata kredytowa - Okresowa wpłata na poczet spłaty kredytu, obejmująca kapitał i
odsetki.
Rata malejąca - System spłat, w którym rata maleje w czasie, przy stałej kwocie
spłacanego kapitału.
Reﬁnansowanie kredytu - Zaciągnięcie nowego kredytu w celu spłaty poprzedniego,
zazwyczaj na korzystniejszych warunkach.
Rejest...
------------------------------------------------------------ 


--- Random example 2 ---
Source: ustawa.pdf
Page: 14
Content (first 500 characters):
Dziennik Ustaw – 15 – Poz. 720 
 
5. Kredytodawca dokonuje restrukturyzacji zadłużenia przez: 
1) zaoferowanie konsumentowi możliwości czasowego zawieszenia spłaty kredytu hipotecznego; 
2) zmianę wysokości rat kapitałowo-odsetko

#### 2. Chunking

In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400, chunk_overlap=50, separators=["\n\n", "\n", ". ", " ", ""]
)
texts = text_splitter.split_documents(documents)

print(f"After splitting: {len(documents)} -> {len(texts)} documents\n")
print(f"Sample chunk:\n{texts[0].page_content}\n\n")

After splitting: 57 -> 592 documents

Sample chunk:
SŁOWNIK POJĘĆ
Nieruchomości i Kredyt Hipoteczny
A
Akt notarialny - Dokument sporządzony przez notariusza, potwierdzający zawarcie
umowy sprzedaży, darowizny lub innej czynności prawnej dotyczącej nieruchomości.
Aktualna wycena nieruchomości - Oszacowanie wartości rynkowej nieruchomości
przez rzeczoznawcę majątkowego, wymagane przez bank przy udzielaniu kredytu
hipotecznego.




#### 3. Embeddings and vector store

In [6]:
embeddings = OpenAIEmbeddings()

# Create Chroma vector store
vectorstore = Chroma.from_documents(
    documents=texts,
    embedding=embeddings,
    collection_name="rag_vs_mortgage_loans",
)

# Check first few embeddings
print("=== SAMPLE EMBEDDING OVERVIEW ===")
sample_text = texts[0].page_content[:100]
print(f"Sample text:\n{sample_text}...\n")

# Get embedding for sample text
sample_embedding = embeddings.embed_query(sample_text)
print(f"Embedding dimension: {len(sample_embedding)}")
print(f"First 5 values: {sample_embedding[:5]}")

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

=== SAMPLE EMBEDDING OVERVIEW ===
Sample text:
SŁOWNIK POJĘĆ
Nieruchomości i Kredyt Hipoteczny
A
Akt notarialny - Dokument sporządzony przez notari...

Embedding dimension: 1536
First 5 values: [-0.007692780811339617, -0.0008024087874218822, 0.026530398055911064, -0.01042727380990982, -0.0037429581861943007]


#### 4. Create RAG retrieval subagent

In [7]:
# Retriever
retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 5,
    }
)

# LLM for RAG
rag_llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

# RAG prompt
rag_template = """
You are RAG tool subagent. 
Based on the following context from official documents, provide a precise answer to the question.

RULES:
- Use ONLY information from the context
- Always include source citations in format: [Source: filename, page: X]
- If answer is not in context, return: "No information available in the provided documents."
- Maintain the same language as the question
- Be factual and cite specific articles/definitions

Context:
{context}

Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(rag_template)


# Format documents with sources
def format_docs(docs):
    formatted = []
    for doc in docs:
        source_info = f"[Source: {doc.metadata['source']}, page {doc.metadata['page']}]"
        formatted.append(f"{doc.page_content}\n{source_info}")
    return "\n\n".join(formatted)


# RAG chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()} | rag_prompt | rag_llm | StrOutputParser()
)

4.1 Testing RAG subagent

In [8]:
test_questions = [
    "Hi, who are you?",
    "what is mortgage?",
    "what is loan?",
    "what's the difference between mortgage and loan?",
    "tell me a joke about banks",
    "jakie są wymagania dotyczące zdolności kredytowej?",
    "co to jest hipoteka?",
    "i earned 5000 USD last month, can I get a loan?",
]

for i, question in enumerate(test_questions, 1):
    print(f"\n\n{'='*60}")
    print(f"TEST {i}/{len(test_questions)}")
    print(f"Question: {question}")
    print(f"{'-'*60}")

    result = rag_chain.invoke(question)

    print(f"Answer: {result}")



TEST 1/8
Question: Hi, who are you?
------------------------------------------------------------
Answer: I am the RAG tool subagent, designed to provide precise answers based solely on the provided official document context, including source citations. How can I assist you?


TEST 2/8
Question: what is mortgage?
------------------------------------------------------------
Answer: Hipoteka to ograniczone prawo rzeczowe na nieruchomości, służące zabezpieczeniu oznaczonej wierzytelności.  
Istnieją różne rodzaje hipoteki, m.in.:  
- hipoteka kaucyjna, która zabezpiecza wierzytelność oznaczoną co do sumy maksymalnej, ale nie co do wysokości,  
- hipoteka łączna, ustanowiona na kilku nieruchomościach dla zabezpieczenia tej samej wierzytelności.  

[Source: slownik_pojec.pdf, page: 2]


TEST 3/8
Question: what is loan?
------------------------------------------------------------
Answer: No information available in the provided documents.


TEST 4/8
Question: what's the difference between m

#### 5. Define RAG as agent tool

In [9]:
@tool
def search_real_estate_knowledge(question: str) -> str:
    """Search the real estate and mortgage loan knowledge base.

    Use this tool to find information about:
    - Mortgage loans and financing
    - Real estate regulations and law
    - Banking procedures
    - Property definitions and terminology

    Args:
        question: The question to search for in the knowledge base

    Returns:
        Answer with source references from official documents
    """
    return rag_chain.invoke(question)

#### 6. Create agent with RAG tool

In [10]:
agent_llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0.0)

agent = create_agent(
    model=agent_llm,
    tools=[search_real_estate_knowledge],
    system_prompt="""

You are a helpful assistant specializing in real estate and mortgage topics.

IMPORTANT RULES:
1. When asked about real estate, mortgages, loans, or banking regulations:
   - USE the search_real_estate_knowledge tool
   - The tool returns answers WITH SOURCE REFERENCES - always include them in your response
   - Present the information clearly, keeping the source citations

2. For general conversation, jokes, or unrelated topics:
   - Answer directly WITHOUT using tools

3. General guidelines:
   - Always respond in the same language as the user's question
   - Be precise and cite sources when discussing regulations or definitions
   - If information is not in the knowledge base, say so clearly

Remember: When you use the search tool, it provides official sources. Always pass these sources to the user!

Expected outputs:
a) When using the search tool:

Answer: (Precise answer based on documents)

[Source: filename, page: X]

b) For general questions:
Answer: (Direct answer without sources)

""",
)

6.1 Testing Agent with RAG subagent as tool

In [11]:
for i, question in enumerate(test_questions, 1):
    print(f"\n{'='*60}")
    print(f"TEST {i}/{len(test_questions)}")
    print(f"Question: {question}")
    print(f"{'-'*60}")

    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    print(f"Answer: {result['messages'][-1].content}")


TEST 1/8
Question: Hi, who are you?
------------------------------------------------------------
Answer: Hello! I am your helpful assistant specializing in real estate, mortgages, loans, and banking regulations. How can I assist you today?

TEST 2/8
Question: what is mortgage?
------------------------------------------------------------
Answer: Answer: A mortgage is a limited real right on real estate, serving as security for a specified claim.

[Source: slownik_pojec.pdf, page: 2]

TEST 3/8
Question: what is loan?
------------------------------------------------------------
Answer: Answer: A loan is a sum of money that is borrowed from a lender, such as a bank or financial institution, which must be repaid with interest over a specified period. Loans are commonly used for various purposes, including purchasing real estate, financing education, or covering personal expenses. The borrower agrees to the terms set by the lender, including the repayment schedule and interest rate.

TEST 4